# Notebook Exploratório: Scraping de Dados de Internações por AVC no RJ via TABNET

Este notebook demonstra o processo de extração e exploração inicial de dados de internações hospitalares por Acidente Vascular Cerebral (AVC) no estado do Rio de Janeiro, utilizando o sistema TABNET do DATASUS.

## Objetivos
- Configurar logging para registrar operações em uma pasta dedicada
- Importar bibliotecas necessárias
- Carregar e processar dados brutos
- Realizar exploração básica dos dados
- Garantir que logs sejam salvos corretamente

Os códigos aqui são baseados no script `explorando.py`, adaptados para execução interativa em um notebook.

## 1. Configuração do Logging

Antes de começar, configuramos o sistema de logging para registrar todas as operações em um arquivo dentro da pasta `logs/`. Isso nos permite acompanhar o progresso e depurar possíveis erros durante o scraping e processamento de dados.

O logging será configurado para salvar mensagens em `logs/execucao.log` com timestamps e níveis de severidade.

In [ ]:
# Configuração do Logging
# Adaptado de log.py: configuramos um logger para salvar em logs/execucao.log
import logging
import sys
import os

# Criar pasta logs se não existir
os.makedirs('logs', exist_ok=True)

# Configurar logger
logger = logging.getLogger('explorando_notebook')
logger.setLevel(logging.DEBUG)

# Formato das mensagens
formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
)

# Handler para arquivo
file_handler = logging.FileHandler("logs/execucao.log", encoding="utf-8")
file_handler.setFormatter(formatter)

# Handler para console
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(formatter)

# Adicionar handlers se não existirem
if not logger.handlers:
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

logger.info("Logging configurado com sucesso. Arquivo: logs/execucao.log")

## 2. Importação de Bibliotecas

Importamos as bibliotecas necessárias para o scraping, processamento de dados e parsing de HTML. Baseado no início do `explorando.py`, usamos `requests` para fazer requisições HTTP, `pandas` para manipulação de dados, `BeautifulSoup` para parsing HTML, e outras utilidades.

In [ ]:
# Importação de Bibliotecas
# Baseado em explorando.py: importamos requests para HTTP, pandas para dados, BeautifulSoup para HTML
import requests
import pandas as pd
import io
from bs4 import BeautifulSoup

logger.info("Bibliotecas importadas com sucesso")

## 3. Carregamento de Dados

Nesta seção, realizamos o scraping dos dados do TABNET. Definimos os cabeçalhos HTTP para simular um navegador, montamos o corpo da requisição POST com os parâmetros específicos para internações por AVC no RJ, fazemos a requisição e processamos a resposta HTML em um DataFrame pandas.

**Nota**: Este código é uma adaptação direta do `explorando.py`, dividido em células para facilitar a execução passo a passo.

In [ ]:
# Definição de Headers HTTP
# Simula uma requisição de navegador para evitar bloqueios
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36",
    "Referer": "https://tabnet.datasus.gov.br/cgi/deftohtm.exe?sih/cnv/nrrj.def",
    "Content-Type": "application/x-www-form-urlencoded",
}

logger.info("Headers HTTP definidos")

In [ ]:
# Corpo da Requisição POST
# Parâmetros para consultar internações por AVC no RJ, baseado em explorando.py
raw_body = (
    "Linha=Munic%EDpio"
    "&Coluna=Ano%2Fm%EAs_processamento"
    "&Incremento=Interna%E7%F5es"
    "&Arquivos=nrrj2602.dbf"
    "&pesqmes1=Digite+o+texto+e+ache+f%E1cil"
    "&SMunic%EDpio=TODAS_AS_CATEGORIAS__"
    "&pesqmes2=Digite+o+texto+e+ache+f%E1cil"
    "&SRegi%E3o_de_Sa%FAde_%28CIR%29=TODAS_AS_CATEGORIAS__"
    "&SMacrorregi%E3o_de_Sa%FAde=TODAS_AS_CATEGORIAS__"
    "&SDivis%E3o_administ_estadual=TODAS_AS_CATEGORIAS__"
    "&pesqmes5=Digite+o+texto+e+ache+f%E1cil"
    "&SMicrorregi%E3o_IBGE=TODAS_AS_CATEGORIAS__"
    "&SRegi%E3o_Metropolitana_-_RIDE=TODAS_AS_CATEGORIAS__"
    "&pesqmes7=Digite+o+texto+e+ache+f%E1cil"
    "&SEstabelecimento=TODAS_AS_CATEGORIAS__"
    "&SCar%E1ter_atendimento=TODAS_AS_CATEGORIAS__"
    "&SRegime=TODAS_AS_CATEGORIAS__"
    "&pesqmes10="
    "&pesqmes11="
    "&SLista_Morb__CID-10=23"  # CID-10 I60-I69 (Doenças Cerebrovasculares)
    "&SLista_Morb__CID-10=177"  # Outro código relacionado
    "&pesqmes12=Digite+o+texto+e+ache+f%E1cil"
    "&SFaixa_Et%E1ria_1=TODAS_AS_CATEGORIAS__"
    "&pesqmes13=Digite+o+texto+e+ache+f%E1cil"
    "&SFaixa_Et%E1ria_2=TODAS_AS_CATEGORIAS__"
    "&SSexo=TODAS_AS_CATEGORIAS__"
    "&SCor%2Fra%E7a=TODAS_AS_CATEGORIAS__"
    "&formato=table"
    "&mostre=Mostra"
)

logger.info("Corpo da requisição POST definido")

In [ ]:
# Fazendo a Requisição POST
# Envia os dados para o TABNET e obtém a tabela HTML
print("Fazendo POST...")
response = requests.post(
    "https://tabnet.datasus.gov.br/cgi/tabcgi.exe?sih/cnv/nrrj.def",
    data=raw_body,
    headers=headers,
    timeout=30
)
response.encoding = "latin-1"  # Define encoding para português

logger.info(f"Requisição concluída. Status: {response.status_code}")

In [ ]:
# Parsing do HTML com BeautifulSoup
# Extrai a tabela da resposta HTML
soup = BeautifulSoup(response.text, "lxml")
tabela = soup.find("table")

logger.info("Tabela HTML extraída")

In [ ]:
# Extração dos Dados da Tabela
# Converte a tabela HTML em uma lista de listas
linhas = tabela.find_all("tr")

dados = []
for linha in linhas:
    celulas = linha.find_all(["td", "th"])
    dados.append([c.get_text(strip=True) for c in celulas])

logger.info(f"Dados extraídos: {len(dados)} linhas")

In [ ]:
# Processamento dos Dados em DataFrame
# Identifica o cabeçalho e cria o DataFrame pandas
idx_header = None
for i, linha in enumerate(dados):
    if linha and linha[0].strip().lower() == "município":
        idx_header = i
        break

if idx_header is None:
    print("Cabeçalho não encontrado, imprimindo primeiras linhas para debug:")
    for l in dados[:5]:
        print(l)
else:
    colunas = dados[idx_header]        # Nomes das colunas
    linhas_dados = dados[idx_header+1:]  # Dados em si

    df = pd.DataFrame(linhas_dados, columns=colunas)

    # Remove linhas completamente vazias
    df = df[df.iloc[:, 0] != ""].reset_index(drop=True)

    logger.info(f"DataFrame criado: {df.shape}")
    print(df.head())
    print(f"Shape: {df.shape}")

## 4. Exploração dos Dados

Com os dados carregados no DataFrame, realizamos uma exploração básica: visualizamos as primeiras linhas, verificamos o tamanho, tipos de dados e estatísticas básicas. Isso nos ajuda a entender a estrutura dos dados antes de prosseguir com análises mais profundas.

In [ ]:
# Exploração Básica dos Dados
# Visualiza informações básicas do DataFrame
print("Informações do DataFrame:")
print(df.info())

print("\nEstatísticas descritivas:")
print(df.describe())

print("\nColunas disponíveis:")
print(df.columns.tolist())

logger.info("Exploração básica concluída")

## 5. Verificação dos Logs

Os logs de todas as operações foram salvos na pasta `logs/`. Podemos verificar o conteúdo do arquivo de log para acompanhar o progresso e identificar possíveis problemas.

In [ ]:
# Verificação dos Logs
# Exibe as últimas linhas do arquivo de log para verificar operações
try:
    with open("logs/execucao.log", "r", encoding="utf-8") as f:
        lines = f.readlines()
        print("Últimas 10 linhas do log:")
        for line in lines[-10:]:
            print(line.strip())
except FileNotFoundError:
    print("Arquivo de log ainda não criado ou não encontrado.")

logger.info("Verificação de logs concluída")

## Conclusão

Este notebook demonstrou o processo completo de scraping e exploração inicial dos dados de internações por AVC no RJ. Os logs foram salvos em `logs/execucao.log` para rastreamento.

Próximos passos: limpeza dos dados, geração de mapas, criação de GIFs de evolução, etc.